# Data Transformation – Gold Layer
## dim_date

This notebook performs the **dimensional modeling transformations** that populate the **gold layer**, generating **business-ready tables** such as dimensions and fact tables in **Delta Lake** format.

In [0]:
CREATE TABLE gold.take_home_test.dim_date (
    date_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    date_id INT,
    date DATE,
    day INT,
    month INT,
    year INT,
    week INT,
    weekday INT,
    is_weekend BOOLEAN,
    inserted_at TIMESTAMP
);

In [0]:
INSERT INTO gold.take_home_test.dim_date (
    date_id, date, day, month, year, week, weekday, is_weekend, inserted_at
)
WITH dates AS (
    SELECT sequence(to_date('2020-01-01'), to_date('2020-12-31'), interval 1 day) AS dt_array
),
exploded AS (
    SELECT explode(dt_array) AS date
    FROM dates
),
iso_calc AS (
    SELECT
        date,

        -- Compute weekday (0=Mon ... 6=Sun)
        ((dayofweek(date) + 5) % 7) AS weekday,

        -- ISO year 
        year(date_add(date, 4 - ((dayofweek(date) + 5) % 7))) AS iso_year,

        -- ISO week
        weekofyear(date) AS iso_week
    FROM exploded
)
SELECT
    CAST(date_format(date, 'yyyyMMdd') AS INT) AS date_id,
    date,
    day(date) AS day,
    month(date) AS month,
    year(date) AS year,

    -- Calculate action_week (INT)
    CAST(
        CONCAT(
            iso_year,
            lpad(
                CASE 
                    -- Primeiros dias antes da primeira semana ISO => 00
                    WHEN iso_week = 1 
                         AND month(date) = 1 
                         AND day(date) <= 5 THEN 0

                    -- The others weeks: ISO_WEEK - 1
                    ELSE iso_week - 1
                END,
                2,
                '0'
            )
        ) AS INT
    ) AS week,

    weekday,
    CASE WHEN weekday IN (5, 6) THEN true ELSE false END AS is_weekend,
    current_timestamp() AS inserted_at
FROM iso_calc
ORDER BY date;